## 说明
这段代码演示了AI Agent的元认知能力，具体展示了一个能够"思考自己的思考"的智能旅行代理。元认知(Metacognition)在AI Agent领域指的是代理能够监控、评估和调整自己的思维过程和行为的能力。

1. **记忆与偏好跟踪**
    - 代理能够记住用户的航班时间偏好
    - 在后续对话中自动应用这些偏好，无需重复询问
    - 明确说明使用了哪些偏好来提供建议
2. **上下文感知对话**
    - 保持对话历史，理解当前对话在整体流程中的位置
    - 根据上下文提供连贯、个性化的响应
    - 能够处理复杂的多轮对话场景
3. **工具调用与函数执行**
    - 演示了如何定义和使用插件工具
    - 展示了流式函数调用的处理过程
    - 可视化显示了内部函数调用详情
4. **自我反思与调整**
    - 当用户表示不满意时，能够调整建议
    - 验证时间框架的合理性并在必要时重新思考
    - 基于反馈持续改进服务质量


## 注意
我在跑的时候发现两个问题
1. 第二次继续对话的时候没有去掉function返回Paris的航班时间信息，直接返回说10:45，但是前面的对话一开始要最晚，后来要最早
2. 反复调用get_flight_times方法获取同一个地区的航班时间信息

In [1]:
import json
import os

from typing import Annotated

from dotenv import load_dotenv

from openai import AsyncOpenAI

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.functions import kernel_function

In [2]:
# 定义目的地插件类 - 包含代理可以调用的工具
# Define a sample plugin for the sample
class DestinationsPlugin:
    """
    提供旅行相关信息的插件，包含目的地列表和航班时间查询功能
    
    根据项目规范"多LLM集成与API兼容性规范"：
    - 工具实现使用@kernel_function装饰器定义
    - 通过Annotated类型注解提供参数描述
    - 将相关工具组织为逻辑单元（Plugin类）
    """

    @kernel_function(description="提供度假目的地的列表。")
    def get_destinations(self) -> Annotated[str, "返回度假目的地。"]:
        """获取可用的度假目的地列表
        
        Returns:
            str: 格式化的目的地列表字符串
        """
        return """
        Barcelona, Spain
        Paris, France
        Berlin, Germany
        Tokyo, Japan
        New York, USA
        """

    @kernel_function(description="提供特定目的地的航班时间。")
    def get_flight_times(
        self, destination: Annotated[str, "要查询航班时间的目的地。"]
    ) -> Annotated[str, "返回指定目的地的航班时间。"]:
        """获取特定目的地的航班时间
        
        Args:
            destination: 目的地名称（可能包含国家）
            
        Returns:
            str: 格式化的航班时间信息
        """
        # 预定义的航班时间数据
        flight_times = {
            "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
            "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
            "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
            "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
            "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
        }

        # Extract just the city name from input that might contain country
        city = destination.split(',')[0].strip()

        if city in flight_times:
            times = ", ".join(flight_times[city])
            return f"{city} 航班时间：{times}"
        else:
            return f"{city} 没有可用的航班信息。"

In [ ]:
load_dotenv()

# 使用通义大模型，作为客户端
model_name = "gpt-4.1-mini"
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)



# 使用GPT大模型，作为客户端
# model_name = "gpt-4o-mini"
# client = AsyncOpenAI(
#     api_key=os.environ["GITHUB_TOKEN"],
#     base_url="https://models.inference.ai.azure.com/"
# )

# 创建AI服务对象，作为Semantic Kernel与GPT模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    # 指定要使用的模型ID（这里是GPT-4.1-mini）
    ai_model_id=model_name,
    # 传入之前创建的AsyncOpenAI客户端
    async_client=client,
)


In [4]:
# 定义代理名称和指令
# 提示词中文翻译如下：
# 您是航班预订代理，提供有关可用航班的信息，并在被询问时提供旅行活动建议。
# 旅行活动建议应针对客户、地点和在地点的时间。

# 您可以使用以下工具来帮助用户计划行程：
# 1. get_destinations：返回用户可以选择的可用度假目的地列表。
# 2. get_flight_times：提供特定目的地的可用航班时间。

# 协助用户的流程：
# - 当用户首次询问没有之前记录的航班预订时，请询问他们首选的航班时间一次。
# - 在整个对话过程中保持 customer_preferences 对象以跟踪首选飞行时间。
# - 当用户预订飞往任何目的地的航班时，在 customer_preferences 对象中记录他们选择的飞行时间。
# - 对于飞往任何目的地的所有后续航班查询，无需询问即可自动应用他们现有的首选飞行时间。
# - 在任何目的地确定时间偏好后，切勿再询问时间偏好。
# - 在建议飞往新目的地的航班时，请明确说：“根据您之前对[时间]航班的偏好，我建议......”
# - 只有在显示与他们喜欢的时间匹配的选项后，才询问他们是否想查看其他时间。
# - 每次预订后，使用任何新信息更新 customer_preferences 对象。
# - 在提出建议时，请务必提及您使用的具体偏好。

# 指引：
# - 使用工具时使用准确的目的地名称（巴塞罗那、巴黎、柏林、东京、纽约）
# - 以乐于助人和热情的方式回应旅行的可能性
# - 始终寻求反馈以确保您的建议满足用户的期望
# - 当请求超出您的能力范围时确认
# - 为了更好地格式化，请始终以列表格式显示飞行时间
# - 在提出任何定时建议时，请考虑时间范围是否合理。如果没有，请再次回复。

# 您的目标是通过了解用户的偏好并提供量身定制的建议，帮助用户有效地探索假期选择并做出明智的旅行决定。
AGENT_NAME = "TravelAgent"
AGENT_INSTRUCTIONS = """ \
你是一名航班预订助手（Flight Booking Agent），负责为用户提供可用航班信息，并在用户提出需求时推荐旅行活动。

推荐的旅行活动应结合**用户特点、目的地以及停留时间**进行个性化建议。

你可以使用以下工具帮助用户规划行程：

1. get_destinations
   返回可供用户选择的度假目的地列表。

2. get_flight_times
   返回指定目的地的可选航班时间。

你的工作流程如下：

- 当用户第一次咨询航班预订且没有任何历史偏好记录时，仅询问一次用户偏好的航班时间。
- 在整个对话过程中，持续维护一个 customer_preferences（用户偏好）对象，用于记录用户偏好的航班时间。
- 当用户预订任意目的地的航班时，将其选择的航班时间记录到 customer_preferences 对象中。
- 此后，对于任何目的地的航班查询，都应自动使用用户已记录的航班时间偏好，而不要再次询问。
- 绝不要在用户已经建立航班时间偏好之后再次询问他们的时间偏好。
- 当为新的目的地推荐航班时，必须明确说明：
  > "根据您之前偏好的【某个时间】航班，我推荐……"
- 在展示符合用户偏好的航班选项之后，再询问用户是否希望查看其他时间段的航班。
- 每次完成预订后，都要更新 customer_preferences 对象，以记录新的用户偏好信息。
- 在给出任何推荐时，都必须明确说明你使用了哪一项用户偏好。

其他要求：

- 调用工具时，必须使用以下精确的目的地名称：
  - Barcelona
  - Paris
  - Berlin
  - Tokyo
  - New York

- 始终以热情、友好且乐于助人的方式回应用户，帮助他们探索旅行可能性。

- 始终主动征求用户反馈，确认你的推荐是否符合他们的需求和期望。

- 当用户提出超出你能力范围的请求时，应明确告知。

- 为了提高可读性，所有航班时间都必须以列表（List）的形式展示。

- 当提供任何涉及时间安排的建议时，应判断时间安排是否合理；如果不合理，应重新给出更合适的建议。

你的目标：

帮助用户高效探索度假目的地，了解他们的旅行偏好，并根据这些偏好提供个性化的航班建议，从而帮助用户做出更明智的旅行决策。

**重要要求：**
你必须使用工具（Tools）来获取和提供航班信息，不能凭空生成航班数据。
"""



# 确保在文件顶部引入了 KernelArguments
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings
#  导入函数选择行为（就是你刚才漏掉的那个！）
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
# 1. 保持你的 execution_settings 定义不变
execution_settings = OpenAIChatPromptExecutionSettings(
    function_choice_behavior=FunctionChoiceBehavior.Auto()
)

# 2. 创建 Agent 时，使用 arguments 替代 execution_settings
# agent = ChatCompletionAgent(
#     service=chat_completion_service,
#     plugins=[DestinationsPlugin()],
#     name=AGENT_NAME,
#     instructions=AGENT_INSTRUCTIONS,
    
# )

# 3. 将设置传递给 Agent  创建AI代理实例
agent = ChatCompletionAgent(
    service=chat_completion_service,  # 使用配置好的聊天服务
    plugins=[DestinationsPlugin()],  # 注册目的地插件
    name=AGENT_NAME,  # 代理名称
    instructions=AGENT_INSTRUCTIONS,  # 代理指令，定义了元认知行为
    arguments=KernelArguments(settings=execution_settings)  # 👈 核心修改点在这里
)

In [5]:
from IPython.display import display, HTML

# 为演示准备的用户输入序列
user_inputs = [
    "帮我预订一张飞往巴塞罗那的机票",
    "我更喜欢晚一点的航班",
    "那个时间还是太晚了，帮我选择最早的航班",
    "我想当天出发，如果我乘坐当天最后一班航班，请给我推荐一些在巴塞罗那中转期间可以做的事情",
    "我有点担心中转时间不够"
]

# Create a thread to hold the conversation
# If no thread is provided, a new thread will be
# created and returned with the initial response
# 创建线程来保存对话历史
# 如果未提供线程，将创建新线程并在初始响应中返回
thread: ChatHistoryAgentThread | None = None

async def main():
    """主函数：演示元认知能力的多轮对话流程
    
    本函数展示了：
    1. 如何维护对话历史（元认知的关键）
    2. 如何处理流式响应和函数调用
    3. 如何可视化显示代理的思考过程
    4. 如何基于上下文提供个性化响应
    
    注意：此示例严格遵循"多LLM集成与API兼容性规范"中的流式响应处理原则
    """
    global thread
    
    # 依次处理每个用户输入
    for user_input in user_inputs:
        # 构建HTML输出（用于在Notebook中美观显示）
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []  # 存储完整响应
        function_calls: list[str] = []  # 存储函数调用详情

        # Buffer to reconstruct streaming function call
        # 用于重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        # 调用代理的流式响应接口
        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread  # 更新对话线程
            agent_name = response.name  # 获取代理名称
            content_items = list(response.items)  # 获取响应内容项

            # 处理每个内容项
            for item in content_items:
                #处理函数调用内容
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    # 累积参数（以流式方式传输）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                # 处理函数结果内容
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    # 完成任何待处理的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            # 尝试解析JSON参数
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                # 处理流式文本内容
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        # ==========================================
        # 🔑 修正版：通过 agent.get_history 异步获取历史
        # ==========================================
        # 此时 invoke_stream 已经结束，Agent 已经把 Tool Call 存入了这个 thread
        # try:
        #     # agent.get_history(thread) 返回的是一个异步迭代器
        #     async for msg in agent.get_history(thread):
        #         # 我们只看最近产生的几条记录（Agent 内部调用的真相就在这里）
        #         # 只有 role 是 assistant (Tool Call) 或 tool (Result) 的我们才关心
        #         if msg.role == "assistant" and hasattr(msg, "items"):
        #             for sub_item in msg.items:
        #                 if sub_item.__class__.__name__ == "FunctionCallContent":
        #                     call_log = f"Calling function: {sub_item.function_name}({sub_item.arguments})"
        #                     if call_log not in function_calls:
        #                         function_calls.append(call_log)
                
        #         elif msg.role == "tool" and hasattr(msg, "items"):
        #             for sub_item in msg.items:
        #                 if sub_item.__class__.__name__ == "FunctionResultContent":
        #                     result_log = f"\nFunction Result:\n\n{sub_item.result}"
        #                     if result_log not in function_calls:
        #                         function_calls.append(result_log)
        # except Exception as e:
        #     print(f"⚠️ 复盘历史记录时出错: {e}")






        # 如果有函数调用，将其添加到HTML输出中
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        # 添加代理响应到HTML输出
        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()


In [6]:
# This will use the same thread that was defined earlier
async def continue_chat():
    """继续对话函数：演示基于历史上下文的对话延续
    
    本函数展示了元认知的另一个关键方面：
    - 如何基于之前的对话历史继续对话
    - 如何应用已学习的用户偏好
    - 如何保持对话的连贯性和上下文感知
    """
    global thread
    
    # Continue the conversation with new user inputs
    # 使用相同的线程继续对话
    user_inputs = [
        "给我订一张去巴黎的机票",
    ]

    for user_input in user_inputs:
        # Start building HTML output
        html_output = "<div style='margin-bottom:10px'>"
        html_output += "<div style='font-weight:bold'>User:</div>"
        html_output += f"<div style='margin-left:20px'>{user_input}</div>"
        html_output += "</div>"

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await continue_chat()


---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
